##    DIM User

# Auto Loader

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_user = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation","abfss://silver@storagespotify2025.dfs.core.windows.net/DimUser/checkpoint")\
                .load("abfss://bronze@storagespotify2025.dfs.core.windows.net/DimUser")


In [0]:
display(df_user)

user_id,user_name,country,subscription_type,start_date,end_date,updated_at,_rescued_data
1,Carlos Berry,Switzerland,Premium,2023-10-17,null,2025-09-23T19:49:55.000Z,null
2,Amanda Jenkins,Montserrat,Family,2024-09-28,null,2025-09-29T19:49:55.000Z,null
3,Daniel Cook,Chile,Premium,2025-07-07,null,2025-09-08T19:49:55.000Z,null
4,Peter Hernandez,Nigeria,Free,2024-07-16,null,2025-09-29T19:49:55.000Z,null
5,Yolanda Morris,Aruba,Premium,2025-05-12,null,2025-09-17T19:49:55.000Z,null
6,Stephen Murphy,Cook Islands,Free,2025-06-17,null,2025-09-12T19:49:55.000Z,null
7,Anthony Andrews Jr.,Libyan Arab Jamahiriya,Free,2024-06-08,null,2025-09-20T19:49:55.000Z,null
8,Felicia Jones,Haiti,Family,2024-09-20,null,2025-09-08T19:49:55.000Z,null
9,Hector Decker,Palau,Family,2024-01-08,null,2025-09-29T19:49:55.000Z,null
10,Frank Davis,New Zealand,Family,2023-12-13,null,2025-09-22T19:49:55.000Z,null


In [0]:
from pyspark.sql.functions import upper, col

df_user_cap = df_user.withColumn(
    "user_name",
    upper(col("user_name"))
)
display(df_user_cap)

In [0]:
df_user = df_user.drop('_rescued_data')
df_user = df_user.dropDuplicates(['user_id'])
display(df_user)

user_id,user_name,country,subscription_type,start_date,end_date,updated_at
148,Chase Moore,Zambia,Family,2024-03-13,null,2025-09-18T19:49:55.000Z
463,Jessica Lewis,Slovenia,Free,2025-03-08,null,2025-10-06T19:49:55.000Z
471,Joshua Butler,Ethiopia,Free,2025-09-14,null,2025-09-25T19:49:55.000Z
496,Andrew Matthews,Taiwan,Family,2025-08-08,null,2025-09-17T19:49:55.000Z
243,Martha Martin,Chad,Premium,2024-05-11,null,2025-10-06T19:49:55.000Z
392,Jennifer Villanueva,Serbia,Free,2025-02-14,null,2025-10-06T19:49:55.000Z
31,Tammy Cook,Bolivia,Premium,2025-02-15,null,2025-10-02T19:49:55.000Z
85,Lisa Sherman,Japan,Family,2024-01-31,null,2025-09-11T19:49:55.000Z
137,Amy Weeks,Sao Tome and Principe,Family,2024-07-02,null,2025-09-09T19:49:55.000Z
251,Jerry Booth,Lithuania,Family,2023-11-04,null,2025-09-17T19:49:55.000Z


In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option(
        "checkpointLocation",
        "abfss://silver@storagespotify2025.dfs.core.windows.net/DimUser/checkpoint"
    ) \
    .trigger(once=True) \
    .option("path","abfss://silver@storagespotify2025.dfs.core.windows.net/DimUser/data") \
    .toTable("spotify_cata.silver.DimUser")

## DimArtist

In [0]:
df_art = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation","abfss://silver@storagespotify2025.dfs.core.windows.net/DimArtist/checkpoint")\
                .load("abfss://bronze@storagespotify2025.dfs.core.windows.net/DimArtist")

display(df_art)

25/11/04 17:36:52 Query aed60b81-84c4-40a9-9d53-81bc91b1dde6 have exception: org.apache.spark.SparkRuntimeException: [STREAMING_STATEFUL_OPERATOR_NOT_MATCH_IN_STATE_METADATA] Streaming stateful operator name does not match with the operator in state metadata. This likely to happen when user adds/removes/changes stateful operator of existing streaming query.
Stateful operators in the metadata: [(OperatorId: 0 -> OperatorName: dedupe), (OperatorId: 1 -> OperatorName: dedupe)]; Stateful operators in current batch: [(OperatorId: 0 -> OperatorName: dedupe)]. SQLSTATE: 42K03
	at org.apache.spark.sql.errors.QueryExecutionErrors$.statefulOperatorNotMatchInStateMetadataError(QueryExecutionErrors.scala:2452)
	at org.apache.spark.sql.execution.streaming.IncrementalExecution$$anon$2.$anonfun$checkOperatorValidWithMetadata$3(IncrementalExecution.scala:805)
	at scala.runtime.java8.JFunction1$mcVJ$sp.apply(JFunction1$mcVJ$sp.scala:18)
	at scala.collection.immutable.Set$Set2.foreach(Set.scala:210)
	at

In [0]:
#Transform data

df_art = df_art.drop('_rescued_data')
df_art = df_art.dropDuplicates(['artist_id'])
display(df_art)

artist_id,artist_name,genre,country,updated_at
148,Dawn Graham,Hip-Hop,Anguilla,2025-10-05T19:49:55.000Z
463,Michael Irwin,Pop,Tokelau,2025-09-23T19:49:55.000Z
471,Alex Mcclure,Jazz,Reunion,2025-09-13T19:49:55.000Z
496,Tim Tucker,Electronic,China,2025-10-05T19:49:55.000Z
243,Leslie Holt,Jazz,Myanmar,2025-09-20T19:49:55.000Z
392,Jason Harrell,Jazz,French Polynesia,2025-10-07T19:49:55.000Z
31,Alexandra Decker,Hip-Hop,Liechtenstein,2025-09-10T19:49:55.000Z
85,Douglas White,Hip-Hop,Antigua and Barbuda,2025-09-15T19:49:55.000Z
137,Charles Perry,Electronic,Zimbabwe,2025-10-04T19:49:55.000Z
251,Mary Moore,Hip-Hop,Bahamas,2025-10-03T19:49:55.000Z


In [0]:
df_art.writeStream.format("delta")\
    .outputMode("append")\
    .option(
        "checkpointLocation",
        "abfss://silver@storagespotify2025.dfs.core.windows.net/DimArtist/checkpoint"
    ) \
    .trigger(once=True) \
    .option("path","abfss://silver@storagespotify2025.dfs.core.windows.net/DimArtist/data") \
    .toTable("spotify_cata.silver.DimArtist")

##Dim Track

In [0]:
df_track = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation","abfss://silver@storagespotify2025.dfs.core.windows.net/DimTrack/checkpoint")\
                .load("abfss://bronze@storagespotify2025.dfs.core.windows.net/DimTrack")

display(df_track)

In [0]:
df_track = df_track.withColumn("durationFlag",when(col('duration_sec')<150,"low")\
    .when(col('duration_sec')<300,"medium")\
        .otherwise("high"))

df_track = df_track.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))

df_track = df_track.drop('_rescued_data')
display(df_track)

In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option(
        "checkpointLocation",
        "abfss://silver@storagespotify2025.dfs.core.windows.net/DimTrack/checkpoint"
    ) \
    .trigger(once=True) \
    .option("path","abfss://silver@storagespotify2025.dfs.core.windows.net/DimTrack/data") \
    .toTable("spotify_cata.silver.DimTrack")

##DIM DATE

In [0]:
df_date = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation","abfss://silver@storagespotify2025.dfs.core.windows.net/DimDate/checkpoint")\
                .load("abfss://bronze@storagespotify2025.dfs.core.windows.net/DimDate") 

display(df_date)

date_key,date,day,month,year,weekday,_rescued_data
20241007,2024-10-07,7,10,2024,Monday,null
20241008,2024-10-08,8,10,2024,Tuesday,null
20241009,2024-10-09,9,10,2024,Wednesday,null
20241010,2024-10-10,10,10,2024,Thursday,null
20241011,2024-10-11,11,10,2024,Friday,null
20241012,2024-10-12,12,10,2024,Saturday,null
20241013,2024-10-13,13,10,2024,Sunday,null
20241014,2024-10-14,14,10,2024,Monday,null
20241015,2024-10-15,15,10,2024,Tuesday,null
20241016,2024-10-16,16,10,2024,Wednesday,null


In [0]:
df_date = df_date.drop('_rescued_data')

df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option(
        "checkpointLocation",
        "abfss://silver@storagespotify2025.dfs.core.windows.net/DimDate/checkpoint"
    ) \
    .trigger(once=True) \
    .option("path","abfss://silver@storagespotify2025.dfs.core.windows.net/DimDate/data") \
    .toTable("spotify_cata.silver.DimDate")

display(df_date)

##FactStream

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation","abfss://silver@storagespotify2025.dfs.core.windows.net/FactStream/checkpoint")\
                .load("abfss://bronze@storagespotify2025.dfs.core.windows.net/FactStream") 

display(df_fact)

In [0]:
df_fact = df_fact.drop('_rescued_data')

df_fact.writeStream.format("delta")\
    .outputMode("append")\
    .option(
        "checkpointLocation",
        "abfss://silver@storagespotify2025.dfs.core.windows.net/FactStream/checkpoint"
    ) \
    .trigger(once=True) \
    .option("path","abfss://silver@storagespotify2025.dfs.core.windows.net/FactStream/data") \
    .toTable("spotify_cata.silver.FactStream")

display(df_fact)

stream_id,user_id,track_id,date_key,listen_duration,device_type,stream_timestamp
1,361,74,20250518,156,Smart Speaker,2025-09-30T19:49:55.000Z
2,321,288,20250519,47,Mobile,2025-09-27T19:49:55.000Z
3,275,340,20250307,214,Mobile,2025-10-03T19:49:55.000Z
4,43,373,20250216,14,Desktop,2025-10-04T19:49:55.000Z
5,319,95,20250421,266,Desktop,2025-09-27T19:49:55.000Z
6,52,31,20250130,317,Desktop,2025-10-02T19:49:55.000Z
7,5,354,20250824,90,Mobile,2025-10-01T19:49:55.000Z
8,115,386,20250413,159,Smart Speaker,2025-09-29T19:49:55.000Z
9,439,95,20250930,290,Mobile,2025-10-04T19:49:55.000Z
10,40,389,20250227,53,Mobile,2025-10-01T19:49:55.000Z


In [0]:
%sql
SELECT *
FROM spotify_cata.gold.dimtrack
WHERE __END_AT IS NOT NULL


track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,durationFlag,__START_AT,__END_AT
46,Expanded foreground knowledgebase,225,Democrat Album,254,2023-09-08,2025-09-14T19:49:55.000Z,medium,2025-09-14T19:49:55.000Z,2025-10-07T19:49:56.000Z
5,Multi layered needs based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,low,2025-10-02T19:49:55.000Z,2025-10-07T19:49:56.000Z


In [0]:
%sql
SELECT *
FROM spotify_cata.gold.dimtrack
WHERE track_id IN (46,5)


track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,durationFlag,__START_AT,__END_AT
46,Team oriented even keeled firmware,9,Owner Album,303,2024-03-22,2025-10-07T19:49:56.000Z,high,2025-10-07T19:49:56.000Z,null
46,Expanded foreground knowledgebase,225,Democrat Album,254,2023-09-08,2025-09-14T19:49:55.000Z,medium,2025-09-14T19:49:55.000Z,2025-10-07T19:49:56.000Z
5,Extended bottom line conglomeration,4,Increase Album,138,2023-01-01,2025-10-07T19:49:56.000Z,low,2025-10-07T19:49:56.000Z,null
5,Multi layered needs based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,low,2025-10-02T19:49:55.000Z,2025-10-07T19:49:56.000Z
